# Multi-station variable extractor

Pick a group, select a variable, and download it from all stations into a tidy DataFrame.
Only 1-D variables (time series without extra dimensions) are offered — multi-dimensional
variables like NEE need coordinate selection first.

**Steps:** Run *Setup* → run *Controls* → choose group/variable → click **Get data**.

In [5]:
import pathlib
import numpy as np
import pandas as pd
import zarr
import xarray as xr
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

STORE = pathlib.Path("icos-fluxnet.zarr")
z     = zarr.open_group(str(STORE), mode="r")

STATIONS = sorted(z.group_keys())
GROUPS   = {
    "root (HH)": "",
    "fluxnet_dd": "fluxnet_dd",
    "fluxnet_mm": "fluxnet_mm",
    "fluxnet_ww": "fluxnet_ww",
    "fluxnet_yy": "fluxnet_yy",
}

def _group_node(station, group_key):
    """Return the zarr group node for a station + group key."""
    return z[station] if group_key == "" else z[station][group_key]

def _common_1d_vars(group_key):
    """Variables present in every station for this group, with exactly 1 dimension."""
    common = None
    for s in STATIONS:
        try:
            node = _group_node(s, group_key)
        except KeyError:
            continue
        arrays = {k for k, v in node.arrays() if len(v.shape) == 1 and v.shape[0] > 1}
        common = arrays if common is None else common & arrays
    return sorted(common or [])

print(f"Store : {STORE}")
print(f"Stations: {len(STATIONS)}  {STATIONS}")

Store : icos-fluxnet.zarr
Stations: 35  ['BE-Bra', 'BE-Dor', 'BE-Lon', 'BE-Maa', 'BE-Vie', 'CZ-BK1', 'CZ-Lnz', 'DE-Geb', 'DE-HoH', 'DE-RuS', 'DE-Tha', 'DK-Sor', 'FI-Hyy', 'FI-Sii', 'FI-Sod', 'FR-Bil', 'FR-FBn', 'FR-Fon', 'FR-Gri', 'FR-Hes', 'FR-Lqu', 'FR-Lus', 'FR-Pue', 'GL-ZaF', 'IT-BCi', 'IT-Cp2', 'IT-MBo', 'IT-Ren', 'IT-SR2', 'NL-Loo', 'NO-Hur', 'SE-Htm', 'SE-Nor', 'SE-Svb', 'UK-AMo']


In [6]:
# ── Widgets ───────────────────────────────────────────────────────────────────

w_group = widgets.Dropdown(
    options=list(GROUPS.keys()),
    value="fluxnet_dd",
    description="Group:",
    layout=widgets.Layout(width="220px"),
)

w_var = widgets.Dropdown(
    options=[],
    description="Variable:",
    layout=widgets.Layout(width="320px"),
)

w_btn = widgets.Button(
    description="Get data",
    button_style="primary",
    layout=widgets.Layout(width="120px"),
)

w_info = widgets.HTML(value="")
out    = widgets.Output()

# Populate variable list when group changes
def _on_group_change(change):
    group_key = GROUPS[change["new"]]
    w_info.value = "<i>Loading variable list…</i>"
    vars_ = _common_1d_vars(group_key)
    w_var.options = vars_
    w_var.value   = vars_[0] if vars_ else None
    w_info.value  = f"<small>{len(vars_)} common 1-D variables</small>"

w_group.observe(_on_group_change, names="value")
_on_group_change({"new": w_group.value})   # initialise

# ── Fetch + plot ──────────────────────────────────────────────────────────────

df_result = None   # last result, accessible after the cell runs

def _decode_times(t_arr):
    """Decode a CF-convention integer time array using its units attribute."""
    t_unit = t_arr.attrs.get("units", "")
    if "since" in t_unit:
        freq, _, origin = t_unit.partition(" since ")
        origin_dt = pd.Timestamp(origin.strip())
        return origin_dt + pd.to_timedelta(t_arr[:].astype(float), unit=freq.strip().rstrip("s"))
    return pd.to_datetime(t_arr[:].astype("datetime64[ns]"))

def _on_get(btn):
    global df_result
    group_key = GROUPS[w_group.value]
    varname   = w_var.value
    if not varname:
        return

    with out:
        clear_output(wait=True)
        print(f"Fetching '{varname}' from {len(STATIONS)} stations …")

        series = {}
        for s in STATIONS:
            try:
                node  = _group_node(s, group_key)
                times = _decode_times(node["time"])
                vals  = node[varname][:].astype(float)
                vals[vals == node[varname].fill_value] = np.nan
                series[s] = pd.Series(vals, index=times, name=s)
            except Exception as exc:
                print(f"  {s}: skipped ({exc})")

        df_result = pd.DataFrame(series)

        # Metadata from first available station
        first = STATIONS[0]
        node0 = _group_node(first, group_key)
        attrs = dict(node0[varname].attrs) if varname in node0 else {}
        long_name = attrs.get("long_name", varname)
        units     = attrs.get("units", "")
        ylabel    = f"{long_name}\n({units})" if units else long_name

        print(f"DataFrame: {df_result.shape[0]} time steps × {df_result.shape[1]} stations")
        display(df_result.head())

        # Plot
        fig, ax = plt.subplots(figsize=(14, 5))
        for col in df_result.columns:
            ax.plot(df_result.index, df_result[col], lw=0.8, alpha=0.7, label=col)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{long_name} — {w_group.value} — all stations")
        ax.legend(fontsize=6, ncol=5, loc="upper right")
        ax.grid(alpha=0.3)
        fig.tight_layout()
        plt.show()

w_btn.on_click(_on_get)

display(
    widgets.HBox([w_group, w_var, w_btn]),
    w_info,
    out,
)

HTML(value='<small>173 common 1-D variables</small>')

Output()

In [8]:
df_result

,BE-Bra,BE-Dor,BE-Lon,BE-Maa,BE-Vie,CZ-BK1,CZ-Lnz,DE-Geb,DE-HoH,DE-RuS,...,IT-Cp2,IT-MBo,IT-Ren,IT-SR2,NL-Loo,NO-Hur,SE-Htm,SE-Nor,SE-Svb,UK-AMo
2018-01-01,NaN,NaN,6.095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5.354,1.755,NaN,NaN
2018-01-02,NaN,NaN,5.830,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,4.502,2.318,NaN,NaN
2018-01-03,NaN,NaN,8.489,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.979,1.557,NaN,NaN
2018-01-04,NaN,NaN,8.473,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2.008,2.167,NaN,NaN
2018-01-05,NaN,NaN,7.747,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.959,0.424,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27,8.104,6.931,7.714,7.183,5.402,2.513,7.224,5.561,4.316,8.428,...,12.440,-0.032,-0.858,9.458,6.200,-6.980,1.718,-4.687,-8.526,4.536
2025-12-28,10.732,8.463,9.460,9.657,7.009,3.460,4.462,8.440,8.678,10.716,...,11.853,-0.113,-1.701,9.824,10.470,-5.361,5.130,-4.288,-12.673,5.643
2025-12-29,10.267,8.265,9.222,9.495,6.990,4.099,6.203,9.444,9.065,10.644,...,12.557,0.005,-1.221,11.083,9.467,-3.520,6.325,-1.035,-6.709,3.945
2025-12-30,8.958,7.664,8.402,8.301,6.493,3.628,7.204,7.742,7.087,9.741,...,12.702,-0.217,-1.482,11.226,7.854,-1.473,4.092,-0.590,-6.901,1.497
